# Lab 08: Investment Comparison
**Question:** How did MSFT, META, AAPL, GOOG and TSLA perform over 2022, and how did they compare to the S&P 500 (`^GSPC`)?

**Data:** daily closing prices from Yahoo Finance. The window starts at the last close of 2021 (Dec 31, 2021) so the return covers all of 2022, through the last close of 2022 (Dec 30, 2022).

**Measures:** total return, volatility, maximum drawdown, and beta/correlation against the S&P 500.

In [ ]:
# Step 1: imports
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
# Step 2: download daily closes (end date is exclusive, so 2023-01-01 includes Dec 30, 2022)
tickers = ["MSFT", "META", "AAPL", "GOOG", "TSLA", "^GSPC"]
prices = yf.download(tickers, start="2021-12-31", end="2023-01-01",
                     auto_adjust=False, progress=False)["Close"]
prices = prices[tickers]   # fixed column order
display(prices.head())
display(prices.tail())

In [ ]:
# Step 3: sanity checks
print("Shape:", prices.shape)
print("First date:", prices.index[0].date(), " Last date:", prices.index[-1].date())
print("Missing values:\n", prices.isna().sum())

Note: Yahoo's `Close` is adjusted for stock splits (GOOG split 20-for-1 in July 2022, TSLA 3-for-1 in August 2022), so there are no artificial price drops. It is not adjusted for dividends, which keeps the stocks comparable to `^GSPC`, a price index.

In [ ]:
# Step 4: growth of $100 invested on Dec 31, 2021
growth = prices / prices.iloc[0] * 100
display(growth.tail())

growth.plot(figsize=(10, 5), title="Growth of $100 over 2022")
plt.axhline(100, color="gray", linestyle="--")
plt.ylabel("Value ($)")
plt.show()

In [ ]:
# Step 5: total return for 2022
total_return = (prices.iloc[-1] / prices.iloc[0] - 1).sort_values()
display(total_return.to_frame("2022 return").style.format("{:.2%}"))

total_return.plot(kind="barh", figsize=(8, 4), title="2022 total return")
plt.axvline(total_return["^GSPC"], color="red", linestyle="--", label="S&P 500")
plt.legend()
plt.show()

In [ ]:
# Step 6: excess return vs the S&P 500 (positive = beat the market)
excess = (total_return - total_return["^GSPC"]).drop("^GSPC").sort_values()
display(excess.to_frame("Return minus S&P 500").style.format("{:.2%}"))

In [ ]:
# Step 7: daily returns
returns = prices.pct_change().dropna()
display(returns.head())
display(returns.tail())

In [ ]:
# Step 8: annualized volatility (risk): daily std x sqrt(252 trading days)
volatility = (returns.std() * 252 ** 0.5).sort_values()
display(volatility.to_frame("Annualized volatility").style.format("{:.2%}"))

In [ ]:
# Step 9: maximum drawdown (worst peak-to-trough fall during the year)
drawdown = prices / prices.cummax() - 1
max_drawdown = drawdown.min().sort_values()
display(max_drawdown.to_frame("Max drawdown").style.format("{:.2%}"))

drawdown.plot(figsize=(10, 5), title="Drawdown from running peak, 2022")
plt.ylabel("Drawdown")
plt.show()

In [ ]:
# Step 10: correlation and beta vs the S&P 500
# beta = cov(stock, market) / var(market): how much the stock moves per 1% market move
market = returns["^GSPC"]
correlation = returns.corr()["^GSPC"]
beta = returns.cov()["^GSPC"] / market.var()
display(returns.corr().round(2))
display(pd.DataFrame({"Correlation": correlation, "Beta": beta}).round(2))

In [ ]:
# Step 11: summary table
summary = pd.DataFrame({
    "2022 return": total_return,
    "vs S&P 500": total_return - total_return["^GSPC"],
    "Volatility": volatility,
    "Max drawdown": max_drawdown,
    "Beta": beta,
}).sort_values("2022 return", ascending=False)
display(summary.style.format({"2022 return": "{:.2%}", "vs S&P 500": "{:.2%}",
                              "Volatility": "{:.2%}", "Max drawdown": "{:.2%}", "Beta": "{:.2f}"}))

## Conclusion
All five stocks lost money in 2022, and all five did worse than the S&P 500.

| | 2022 return | vs S&P 500 | Volatility | Max drawdown | Beta |
|---|---|---|---|---|---|
| S&P 500 | -19.44% | | 24.17% | -25.43% | 1.00 |
| AAPL | -26.83% | -7.39% | 35.70% | -30.75% | 1.31 |
| MSFT | -28.69% | -9.25% | 35.34% | -36.30% | 1.28 |
| GOOG | -38.67% | -19.23% | 38.77% | -43.60% | 1.34 |
| META | -64.22% | -44.78% | 64.26% | -73.74% | 1.74 |
| TSLA | -65.03% | -45.59% | 66.68% | -72.72% | 1.75 |

- **Two groups.** AAPL and MSFT trailed the market by 7 to 9 points, roughly what their betas (about 1.3) predict in a falling market. META and TSLA lost about two-thirds of their value and fell more than 70% from peak at some point in the year.
- **Higher risk, bigger losses.** The ranking by volatility and beta lines up almost exactly with the ranking by loss. In a down year, high-beta growth stocks amplified the market decline instead of protecting against it.
- **Diversification helped.** The S&P 500 had the lowest volatility (24%) and the smallest drawdown (-25%). Even the best of the five, AAPL, was about 1.5 times as volatile as the index.
- **Why 2022:** the Fed raised rates aggressively to fight inflation, which hit long-duration growth stocks hardest. META also faced spending concerns over its metaverse push, and TSLA faced demand worries and CEO distraction.

**Limitations:** price returns only (dividends excluded, but AAPL and MSFT yields were under 1%, so the ranking would not change); a single calendar year is a short window; beta and volatility are estimated from 251 daily returns.